# 05 · Formato de storage do lake — `.duckdb` vs DuckLake vs Delta

**A pergunta central da arquitetura de dados.** Hoje RAW = Delta e **CLEAN = `.duckdb`**
(arquivo monolítico). O `.duckdb` trava o MinIO como lake e reencena o single-writer na
borda CLEAN→grafo. Este notebook consolida as 4 investigações que rodaram esse tema:

| Seção | Pergunta | Veredito |
|---|---|---|
| §1 MinIO | `.duckdb` funciona sobre `s3://`? | 🛑 só read-only — é o bloqueio |
| §2 DuckLake | pipeline inteiro sem `.duckdb`? | ✅ dlt + dbt **nativos**, catálogo SQL |
| §3 Delta | pipeline inteiro em Delta até o grafo? | ✅ roda; dbt precisa de ponte OU plugin |
| §4 dbt→Delta | dbt escreve Delta direto (sem ponte)? | ✅ via plugin `store()`; write duplo inevitável |

> 💡 **DuckDB é engine, não storage.** O `.duckdb` é um formato de arquivo *opcional* (como
> o `.sqlite`). Trocá-lo por Delta/DuckLake mantém a engine e abre o storage. Resultados
> completos em [RESULTADOS.md](RESULTADOS.md). Decisão em aberto:
> [pontos-a-verificar §1](../../docs/arquitetura/2.0-lake-aberto/pontos-a-verificar.md).

> ⚠️ Requer FalkorDB em `localhost:6380` pra §3 (`docker run -p 6380:6379 falkordb/falkordb`)
> e um MinIO pra §1 sobre S3 real (opcional — em disco também demonstra).

In [ ]:
import sys; sys.path.insert(0, "/Users/allanfraga/Repos/strattum/experimentacoes")
import exp
import duckdb, deltalake, dlt
print("duckdb", duckdb.__version__, "| dlt", dlt.__version__, "| deltalake", deltalake.__version__)

## §1 · MinIO — o `.duckdb` não vai pro object store
RAW (Delta/Parquet) vai pro `s3://` sem drama (Polars e DuckDB leem/escrevem sem cópia).
O **bloqueio** é o `.duckdb` da CLEAN: sobre `s3://` só abre **read-only** — o formato nativo
do DuckDB precisa de filesystem real (lock/append). Foi isso que motivou DuckLake e Delta.

In [ ]:
# RAW em Delta sobre s3:// — OK (precisa de MinIO + AWS_* no ambiente)
import polars as pl
raw = f"{exp.DATA}/minio_demo/raw_orders"   # troque por s3://bucket/raw_orders contra MinIO
pl.DataFrame({"id": range(100), "amount": [i*1.5 for i in range(100)]}).write_delta(raw, mode="overwrite")
print("RAW Delta linhas:", exp.count_delta(raw))

# O .duckdb sobre s3:// -> só read-only. Escrita falha (por isso saímos do .duckdb).
# con = duckdb.connect('s3://bucket/clean.duckdb')  # IOException em escrita

## §2 · DuckLake — pipeline inteiro, sem `.duckdb`
DuckLake = **dados em Parquet + catálogo SQL** (DuckDB/Postgres). dlt tem destination
`ducklake`; o dbt-duckdb dá `attach` e materializa **nativo** no lake (`--full-refresh` =
overwrite; `run` = incremental). **Zero gambiarra** — é o candidato mais simples.

Modelo dbt de referência: [`models-reference/clean_orders_ducklake.sql`](models-reference/clean_orders_ducklake.sql).

In [ ]:
# Esqueleto (a mecânica rodou em disco; contra o starter é trocar a config p/ Postgres+MinIO):
#   ATTACH 'ducklake:<catalogo>' AS lake;  -- catálogo em DuckDB/Postgres, dados em Parquet
#   dlt.pipeline(destination='ducklake', ...).run(source, write_disposition='replace')  # RAW
#   dbt run --full-refresh   # CLEAN nativa no lake (overwrite);  dbt run  -> incremental
con = duckdb.connect()
con.execute("INSTALL ducklake; LOAD ducklake;")
print("extensão ducklake OK — attach do catálogo materializa a CLEAN direto no lake")

## §3 · Delta em tudo (RAW + CLEAN) — roda até o grafo
Pipeline inteiro em Delta, até o FalkorDB. O script completo (dlt → RAW Delta → dbt +
`write_deltalake` → CLEAN Delta → Cypher → FalkorDB, overwrite **e** incremental) está em
[`delta_pipeline.py`](delta_pipeline.py). **Aprendizado:** o `dbt-duckdb` **não escreve Delta**
— materializa numa tabela scratch e uma **ponte `write_deltalake`** grava o Delta (ver §4 p/ alternativa).

In [ ]:
# roda o pipeline Delta completo (requer FalkorDB em :6380)
import subprocess, sys
subprocess.run([sys.executable, "delta_pipeline.py"], cwd="05-formato-storage-lake")
# saída esperada: RAW 100->120, CLEAN 100->120, nós :Order no FalkorDB = 120 ✅

## §4 · O dbt escreve Delta direto? (elimina a ponte externa)
Sim — via **plugin custom** [`dbt-delta-plugin/delta_writer.py`](dbt-delta-plugin/delta_writer.py) que implementa `store()`,
o hook que a materialização `external` chama **dentro do `dbt run`**. Um único `dbt run` faz
overwrite **e** merge/upsert, sem tabela `.duckdb` da clean e sem passo Python no flow.

🛑 **Caveat honesto:** `external` sempre grava um parquet físico (`COPY TO`) antes de chamar
`store()` — não existe `COPY TO (FORMAT delta)` no DuckDB. Então há **write duplo** (parquet
temp → Delta), inevitável hoje. O parquet temp é apagado; o artefato final é Delta puro.
Se o critério é *zero write duplo / sem ponte nenhuma*, **DuckLake ainda vence** (§2).

In [ ]:
# teste A/B (overwrite -> 100, merge -> 115) do plugin dbt->Delta
import subprocess, sys
subprocess.run([sys.executable, "dbt_delta_test.py"], cwd="05-formato-storage-lake")

## Migração (quando decidir)
O backfill `.duckdb` → lake (Delta **ou** DuckLake) e o cutover dos leitores estão em
[`migrate_duckdb_to_lake.py`](migrate_duckdb_to_lake.py) + runbook
[docs/arquitetura/migracao-lake-aberto.md](../../docs/arquitetura/migracao-lake-aberto.md).